<!-- NB_DOC_INTRO_V1 -->
# Systemes de recommandation

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Les **systemes de recommandation** predisent ce qu'un utilisateur va aimer/cliquer/acheter a partir de son historique et/ou du contenu des items. C'est l'un des cas d'usage ML les plus rentables (Amazon : ~30% revenue, Netflix : 80% des visionnages, Spotify : Discover Weekly).

Ce notebook implemente, **avec code execute**, les **5 familles principales** :
1. **Popularite** (baseline incontournable)
2. **Content-based** (similarite TF-IDF / embeddings)
3. **Collaborative filtering** user-based et item-based (KNN cosinus)
4. **Matrix Factorization** (SVD, ALS, NMF)
5. **Hybrid** (combinaison content + collaborative)

Avec un **MovieLens 100k** synthetique reproduit (ratings 1-5) et un dataset implicite (clics).

Cover aussi : **metriques specifiques recsys** (Precision@k, Recall@k, NDCG@k, MAP@k, MRR, HitRate@k, Coverage, Diversity), **split temporel** (PAS de KFold random !), et les **pieges typiques** (cold-start, position bias, popularity bias, filter bubble).

Versions : `numpy`, `scipy`, `sklearn`, optionnellement `implicit`.

## Plan

1. Datasets jouet (explicite et implicite)
2. **Baseline popularite**
3. **Content-based** (TF-IDF + cosine similarity)
4. **Collaborative filtering** user-based + item-based (sklearn cosine)
5. **Matrix factorization** (SVD truncated + NMF)
6. **ALS pour implicit feedback** (implicit lib)
7. **Metriques recsys** (Precision@k, NDCG@k, MAP@k, MRR, Coverage)
8. **Split temporel** (eviter le data leakage)
9. Comparatif final
10. Pieges et anti-patterns
11. References


## 1. Datasets jouet

On simule un dataset style **MovieLens** :
- N users (100), M items (50)
- Ratings explicites 1-5 (sparse — chaque user note ~10 items)
- Cible : predire les ratings manquants, ou recommander top-k items


In [ ]:
# pip install -q numpy pandas scipy scikit-learn matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# === Generation MovieLens-like explicite ===
n_users, n_items = 100, 50

# Vrais facteurs latents (ground truth pour evaluer)
true_user_factors = np.random.randn(n_users, 5)
true_item_factors = np.random.randn(n_items, 5)
true_ratings = true_user_factors @ true_item_factors.T

# Normaliser dans [1, 5]
true_ratings = 1 + 4 * (true_ratings - true_ratings.min()) / (true_ratings.max() - true_ratings.min())

# Sparse observations : chaque user note ~10 items au hasard
mask = np.random.rand(n_users, n_items) < 0.2   # ~20% observed
obs_ratings = np.where(mask, true_ratings + np.random.randn(n_users, n_items) * 0.3, 0)
obs_ratings = np.clip(obs_ratings, 1, 5)

# Long format pour traitement
events = []
for u in range(n_users):
    for i in range(n_items):
        if obs_ratings[u, i] > 0:
            events.append({"user": u, "item": i, "rating": obs_ratings[u, i]})
ratings_df = pd.DataFrame(events)
print(f"Total observations : {len(ratings_df)}, density : {len(ratings_df) / (n_users * n_items):.2%}")
ratings_df.head()

### 1.1 Matrice sparse user-item (CSR)


In [ ]:
# Matrice sparse explicit (rating)
ui_matrix = sparse.csr_matrix(
    (ratings_df["rating"], (ratings_df["user"], ratings_df["item"])),
    shape=(n_users, n_items),
)
print(f"User-item matrix : {ui_matrix.shape}, nnz = {ui_matrix.nnz}")

# Matrice implicit (binaire : a interagit ou pas)
ui_implicit = (ui_matrix > 0).astype(np.float32)
print(f"Implicit matrix density : {ui_implicit.nnz / (n_users * n_items):.2%}")

### 1.2 Train/test split — TEMPOREL (anti-leakage)

**Anti-pattern critique** : splitter aleatoirement les ratings. Ca fait fuiter le futur dans le train.
La bonne approche : ajouter un `timestamp`, trier par temps, garder les derniers 20% pour le test.

Pour la demo on simule un timestamp aleatoire (substitut OK pour montrer la mecanique).


In [ ]:
ratings_df["ts"] = np.random.rand(len(ratings_df))   # substitut timestamp
ratings_df = ratings_df.sort_values("ts").reset_index(drop=True)

split = int(0.8 * len(ratings_df))
train_df = ratings_df.iloc[:split]
test_df = ratings_df.iloc[split:]

print(f"Train : {len(train_df)} ratings, Test : {len(test_df)} ratings")

# Re-creer la matrice train uniquement
ui_train = sparse.csr_matrix(
    (train_df["rating"], (train_df["user"], train_df["item"])),
    shape=(n_users, n_items),
)

## 2. Baseline — popularite

**Toujours** commencer par cette baseline. Recommander a chacun les items les plus populaires globalement.

> 🎯 Souvent suffit a 60-70% de la qualite du best model — important de comparer dessus.


In [ ]:
# Popularite = nb d'interactions par item dans train
item_popularity = np.asarray(ui_train.sum(axis=0)).ravel()

def recommend_popularity(user_id, k=10, exclude_seen=True):
    seen = set(ui_train.getrow(user_id).indices)
    scores = item_popularity.copy()
    if exclude_seen:
        scores[list(seen)] = -np.inf
    top_k = np.argsort(scores)[-k:][::-1]
    return top_k

# Recommandations pour user 0
print(f"User 0 a vu : {sorted(ui_train.getrow(0).indices)[:10]}...")
print(f"Top 10 recommandes (popularite) : {recommend_popularity(0, k=10)}")

## 3. Content-based — similarite TF-IDF

On suppose qu'on a des **features texte** sur les items (titre, description, tags). Pour chaque user, on construit son **profil** = moyenne (ponderee par rating) des items qu'il a aimes. On recommande les items les plus similaires a ce profil.

### 3.1 Generer du contenu items synthetique


In [ ]:
# Generer des "tags" par item (3-5 mots issus d'un vocabulaire de 20)
np.random.seed(SEED)
vocab = ["action", "comedy", "drama", "thriller", "scifi", "romance", "horror",
         "documentary", "animation", "biography", "war", "western", "musical",
         "crime", "fantasy", "mystery", "history", "sport", "adventure", "family"]

item_tags = []
for i in range(n_items):
    n_tags = np.random.randint(3, 6)
    tags = " ".join(np.random.choice(vocab, n_tags, replace=False))
    item_tags.append(tags)

# Aperçu
for i in range(3):
    print(f"Item {i} : {item_tags[i]}")

In [ ]:
# TF-IDF des tags
tfidf = TfidfVectorizer()
item_features = tfidf.fit_transform(item_tags)
print(f"Item features shape : {item_features.shape}")

# Pour chaque user : profil = moyenne (ponderee par rating) des items vus
def user_profile(user_id):
    user_row = ui_train.getrow(user_id)
    seen_items = user_row.indices
    if len(seen_items) == 0:
        return None
    seen_features = item_features[seen_items]      # sparse
    seen_ratings = user_row.data
    # Moyenne ponderee
    profile = seen_features.multiply(seen_ratings[:, None]).sum(axis=0) / seen_ratings.sum()
    return np.asarray(profile)

def recommend_content(user_id, k=10):
    profile = user_profile(user_id)
    if profile is None:
        return recommend_popularity(user_id, k=k)
    sims = cosine_similarity(profile, item_features.toarray()).ravel()
    seen = set(ui_train.getrow(user_id).indices)
    sims[list(seen)] = -np.inf
    return np.argsort(sims)[-k:][::-1]

print(f"Top 10 (content-based) pour user 0 : {recommend_content(0, k=10)}")

## 4. Collaborative Filtering — KNN cosinus

### 4.1 User-based : "les users similaires a toi ont aime X"


In [ ]:
# Similarite user-user (cosinus sur la matrice de ratings)
# Normaliser par utilisateur (eliminer le biais "user toujours genereux")
def normalize_rows(mat):
    mat = mat.toarray().astype(float)
    means = np.where(mat.sum(axis=1) > 0,
                     mat.sum(axis=1) / np.maximum((mat > 0).sum(axis=1), 1), 0)
    centered = np.where(mat > 0, mat - means[:, None], 0)
    return centered

centered = normalize_rows(ui_train)
user_sim = cosine_similarity(centered)

def recommend_user_cf(user_id, k=10, n_neighbors=10):
    sims = user_sim[user_id].copy()
    sims[user_id] = 0  # exclude self
    top_neighbors = np.argsort(sims)[-n_neighbors:]

    # Score predit = moyenne ponderee par sim des ratings des voisins
    scores = np.zeros(n_items)
    weight_sum = np.zeros(n_items)
    for nbr in top_neighbors:
        nbr_ratings = ui_train.getrow(nbr).toarray().ravel()
        mask = nbr_ratings > 0
        scores[mask] += sims[nbr] * nbr_ratings[mask]
        weight_sum[mask] += sims[nbr]
    pred = np.where(weight_sum > 0, scores / weight_sum, 0)

    seen = set(ui_train.getrow(user_id).indices)
    pred[list(seen)] = -np.inf
    return np.argsort(pred)[-k:][::-1]

print(f"Top 10 (user CF) pour user 0 : {recommend_user_cf(0, k=10)}")

### 4.2 Item-based : "ceux qui ont aime X ont aussi aime Y"

Item-based est souvent **plus stable** (les items changent moins que les users) et **plus scalable** (on precalcule sim item-item une fois).


In [ ]:
# Similarite item-item
item_sim = cosine_similarity(ui_train.T)

def recommend_item_cf(user_id, k=10):
    user_ratings = ui_train.getrow(user_id).toarray().ravel()
    seen_items = np.where(user_ratings > 0)[0]
    if len(seen_items) == 0:
        return recommend_popularity(user_id, k=k)

    # Score = somme ponderee par rating user × sim item-item
    scores = item_sim[seen_items].T @ user_ratings[seen_items]
    scores[seen_items] = -np.inf
    return np.argsort(scores)[-k:][::-1]

print(f"Top 10 (item CF) pour user 0 : {recommend_item_cf(0, k=10)}")

## 5. Matrix Factorization — SVD et NMF

L'idee : decomposer `R ≈ U × V^T` ou `U` (n_users, k) et `V` (n_items, k) sont des **embeddings latents** de petite dimension. Les ratings predits = `U_u · V_i^T`.

**Truncated SVD** : decomposition algebrique, rapide. **NMF** : contraintes positives (interpretable comme "topics").


In [ ]:
# SVD truncated
svd = TruncatedSVD(n_components=10, random_state=SEED)
U_svd = svd.fit_transform(ui_train)
V_svd = svd.components_.T

# Predictions de rating
pred_svd = U_svd @ V_svd.T

# Eval sur test
test_preds = [pred_svd[r.user, r.item] for r in test_df.itertuples()]
test_truth = test_df["rating"].values

rmse_svd = np.sqrt(np.mean((np.array(test_preds) - test_truth) ** 2))
print(f"SVD truncated : RMSE test = {rmse_svd:.4f}")

In [ ]:
# NMF (factors >= 0)
nmf = NMF(n_components=10, init="random", random_state=SEED, max_iter=200)
U_nmf = nmf.fit_transform(ui_train)
V_nmf = nmf.components_.T

pred_nmf = U_nmf @ V_nmf.T
test_preds_nmf = [pred_nmf[r.user, r.item] for r in test_df.itertuples()]
rmse_nmf = np.sqrt(np.mean((np.array(test_preds_nmf) - test_truth) ** 2))
print(f"NMF : RMSE test = {rmse_nmf:.4f}")

### 5.1 Recommandation top-k via embeddings appris


In [ ]:
def recommend_svd(user_id, k=10):
    scores = U_svd[user_id] @ V_svd.T
    seen = set(ui_train.getrow(user_id).indices)
    scores[list(seen)] = -np.inf
    return np.argsort(scores)[-k:][::-1]

print(f"Top 10 (SVD) pour user 0 : {recommend_svd(0, k=10)}")

## 6. ALS pour implicit feedback

Quand on n'a pas de ratings explicites, juste des **clicks / vues / achats** (binaire). L'**ALS** (Alternating Least Squares) avec le **weight scheme** de Hu et al. 2008 est le standard.

Si la lib `implicit` est installee → code execute. Sinon, on tombe sur la version "fallback" SVD sur la matrice binaire.


In [ ]:
# pip install -q implicit
try:
    from implicit.als import AlternatingLeastSquares
    from implicit.bpr import BayesianPersonalizedRanking
    import os
    os.environ["OPENBLAS_NUM_THREADS"] = "1"   # eviter conflit threads

    # ALS attend une matrice user-item sparse
    als = AlternatingLeastSquares(factors=20, regularization=0.01, iterations=15, random_state=SEED)
    als.fit(ui_implicit)

    # Recommandations pour user 0
    ids, scores = als.recommend(0, ui_implicit[0], N=10)
    print(f"Top 10 (ALS implicit) pour user 0 : {ids.tolist()}")
    print(f"Scores : {scores.tolist()}")

    # Items similaires a l'item 0
    sim_ids, sim_scores = als.similar_items(0, N=5)
    print(f"\nItems similaires a item 0 : {sim_ids.tolist()}")
except ImportError:
    print("implicit pas installe. Fallback : SVD sur la matrice binaire.")
    svd_imp = TruncatedSVD(n_components=10, random_state=SEED)
    U_imp = svd_imp.fit_transform(ui_implicit)
    V_imp = svd_imp.components_.T
    scores = U_imp[0] @ V_imp.T
    print(f"Top 10 (SVD implicit fallback) : {np.argsort(scores)[-10:][::-1]}")

## 7. Metriques recsys

Voir [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) section 6 pour le detail. On implemente ici les principales et on les calcule pour chaque algo.


In [ ]:
def precision_at_k(relevant, ranked, k):
    return np.mean([item in relevant for item in ranked[:k]])

def recall_at_k(relevant, ranked, k):
    if len(relevant) == 0: return 0.0
    return sum(1 for item in ranked[:k] if item in relevant) / len(relevant)

def hit_rate_at_k(relevant, ranked, k):
    return float(any(item in relevant for item in ranked[:k]))

def average_precision_at_k(relevant, ranked, k):
    if not relevant: return 0.0
    score = 0.0; n_hits = 0
    for i, item in enumerate(ranked[:k], start=1):
        if item in relevant:
            n_hits += 1
            score += n_hits / i
    return score / min(len(relevant), k)

def ndcg_at_k(relevant, ranked, k):
    """Normalized DCG : recompense les bons items en haut du classement."""
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(ranked[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def reciprocal_rank(relevant, ranked):
    for i, item in enumerate(ranked, start=1):
        if item in relevant: return 1.0 / i
    return 0.0

# Demo sur user 0
relevant_user0 = set(test_df[test_df["user"] == 0]["item"].tolist())
ranked_user0 = list(recommend_user_cf(0, k=20))
print(f"User 0 relevant items (in test) : {relevant_user0}")
print(f"User 0 ranked (user CF)         : {ranked_user0[:10]}")
print()
for k in [5, 10, 20]:
    print(f"  k={k:2d}  P={precision_at_k(relevant_user0, ranked_user0, k):.3f}"
          f"  R={recall_at_k(relevant_user0, ranked_user0, k):.3f}"
          f"  NDCG={ndcg_at_k(relevant_user0, ranked_user0, k):.3f}"
          f"  MAP={average_precision_at_k(relevant_user0, ranked_user0, k):.3f}"
          f"  Hit={hit_rate_at_k(relevant_user0, ranked_user0, k):.0f}")
print(f"MRR : {reciprocal_rank(relevant_user0, ranked_user0):.3f}")

### 7.1 Evaluation globale (moyenne sur tous les users)


In [ ]:
def evaluate(recommend_fn, k=10):
    """Evalue un recommender sur le test set."""
    precs, recalls, ndcgs, hits, rrs = [], [], [], [], []
    test_users = test_df["user"].unique()

    all_recommended = set()
    for u in test_users:
        relevant = set(test_df[test_df["user"] == u]["item"].tolist())
        if not relevant: continue
        ranked = list(recommend_fn(u, k=k))
        all_recommended.update(ranked)

        precs.append(precision_at_k(relevant, ranked, k))
        recalls.append(recall_at_k(relevant, ranked, k))
        ndcgs.append(ndcg_at_k(relevant, ranked, k))
        hits.append(hit_rate_at_k(relevant, ranked, k))
        rrs.append(reciprocal_rank(relevant, ranked))

    coverage = len(all_recommended) / n_items
    return {
        "P@k":      np.mean(precs),
        "R@k":      np.mean(recalls),
        "NDCG@k":   np.mean(ndcgs),
        "HitRate":  np.mean(hits),
        "MRR":      np.mean(rrs),
        "Coverage": coverage,
    }

# Comparatif
algos = {
    "Popularity":    lambda u, k: recommend_popularity(u, k),
    "Content-based": lambda u, k: recommend_content(u, k),
    "User CF":       lambda u, k: recommend_user_cf(u, k),
    "Item CF":       lambda u, k: recommend_item_cf(u, k),
    "SVD":           lambda u, k: recommend_svd(u, k),
}
results = pd.DataFrame({name: evaluate(fn, k=10) for name, fn in algos.items()}).T
print("Evaluation @ k=10 :")
results.round(4)

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Split random KFold sur events temporels | Split temporel (sort by ts + last 20% test) |
| Evaluer sur tous les items sans exclure les "deja vus" | Exclure items dans `seen` |
| Optimiser RMSE quand le business veut top-k clicks | Utiliser NDCG@k / MAP@k |
| Ignorer le **position bias** (items en haut plus cliques) | Bandits contextuels / debiaising |
| **Cold-start** non traite (nouveau user / item) | Fallback popularite / content-based / metadata |
| Toujours recommander les blockbusters | Mesurer **Coverage** + **Diversity** + reranker MMR |
| Pas de baseline popularite | Toujours comparer (gain reel != gain naif) |
| Recommander des items deja achetes (groceries) | Selon contexte, garder est OK |
| Confondre **explicit** (rating) et **implicit** (clic) | Algo et metriques differents (RMSE vs MAP@k) |
| Single-armed approach | Considerer bandits / online learning pour A/B continu |


## 9. References

### Documentation officielle
- **Surprise** : http://surpriselib.com/
- **implicit** : https://benfred.github.io/implicit/
- **LightFM** (hybride) : https://making.lyst.com/lightfm/docs/home.html
- **RecBole** : https://recbole.io/
- **NVIDIA Merlin** : https://github.com/NVIDIA-Merlin/Merlin
- **Cornac** : https://cornac.preferred.ai/

### Papers fondateurs
- Hu, Koren, Volinsky (2008). *Collaborative Filtering for Implicit Feedback Datasets* (ALS implicit).
- Koren, Bell, Volinsky (2009). *Matrix Factorization Techniques for Recommender Systems*.
- Rendle et al. (2009). *BPR: Bayesian Personalized Ranking from Implicit Feedback*.
- He et al. (2017). *Neural Collaborative Filtering*.
- Sun et al. (2019). *BERT4Rec*.
- He et al. (2020). *LightGCN* (graph-based recsys).

### Voir aussi (collection)
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — metriques recsys detaillees (section 6)
- [BDD_Vectorielles.ipynb](./BDD_Vectorielles.ipynb) — FAISS pour retrieval top-k
- [ML_Apprentissage_par_Renforcement.ipynb](./ML_Apprentissage_par_Renforcement.ipynb) — bandits contextuels
